# Open Buildings Explorer BR — Notebook Demonstrativo

Este notebook mostra como usar a **API Python** do `obr_explorer` sem passar pela CLI.
Útil para quem quer entender como o pipeline funciona por dentro, integrar o projeto
em um script maior, ou prototipar análises customizadas.

**Pré-requisitos**: ter rodado `pip install -e .` na raiz do repositório.

**Tempo de execução**: a primeira execução contra o S3 real demora 30-45 min (download
de 141M footprints do Brasil). Execuções subsequentes com `--pular-carga` são de segundos.

## Sumário

1. [Setup](#1-setup)
2. [Explorando AOIs disponíveis](#2-explorando-aois-disponíveis)
3. [Conexão DuckDB e extensões](#3-conexão-duckdb-e-extensões)
4. [Pipeline: carrega país → recorta por AOI → analisa](#4-pipeline-carrega-país--recorta-por-aoi--analisa)
5. [Comparação entre AOIs](#5-comparação-entre-aois)
6. [Exportação e visualização](#6-exportação-e-visualização)


## 1. Setup

In [ ]:
import logging
from pathlib import Path

# Importar os módulos principais do pacote
from obr_explorer import aoi as aoi_mod
from obr_explorer import analysis
from obr_explorer import config
from obr_explorer import duckdb_client as ddb
from obr_explorer import webmap

# Configura logging para ver o progresso
config.configure_logging(level='INFO')

## 2. Explorando AOIs disponíveis

As AOIs são GeoJSONs versionados em `data/external/aois/`.

In [ ]:
# Lista todas as AOIs disponíveis
aois = aoi_mod.listar_aois()
for a in aois:
    props = a['properties']
    print(f"{a['nome']:<20} {props.get('nome', '?'):<15} IBGE={props.get('ibge_code', '?')}")

In [ ]:
# Carregar uma AOI específica
cambuquira = aoi_mod.carregar_aoi('cambuquira_mg')
print(f"Nome: {cambuquira.nome_exibicao}")
print(f"Bounds (WGS84): {cambuquira.bounds}")
print(f"Área oficial IBGE: {cambuquira.properties['area_km2_oficial']:.3f} km²")
print(f"População 2022: {cambuquira.properties['populacao_2022']:,} hab")
print(f"Região intermediária: {cambuquira.properties.get('regiao_intermediaria')}")

## 3. Conexão DuckDB e extensões

A extensão `spatial` dá acesso a funções `ST_*` e `httpfs` permite ler Parquet diretamente do S3 público.

In [ ]:
# Criar conexão (in-memory por default — rápido, dados somem ao fechar)
# Para persistir e reusar entre runs: ddb.criar_conexao(in_memory=False, database_path=Path('dados.duckdb'))
con = ddb.criar_conexao()

# Instalar e carregar extensões (primeira vez baixa do CDN, depois fica em cache)
ddb.instalar_extensoes(con)

# Configurar S3 (anônimo — bucket VIDA é público; configura path-style URL + CA bundle)
ddb.configurar_s3(con)

# Confirmar que tudo está carregado
ddb.extensoes_carregadas(con)

## 4. Pipeline: carrega país → recorta por AOI → analisa

A operação crítica aqui — download do Brasil inteiro — demora **~30-45 min na primeira vez**. Se você já tem um banco persistente com `footprints_bra` carregado, use `in_memory=False` e `database_path` na célula acima e esta célula vira instantânea nas próximas execuções.

In [ ]:
# ATENÇÃO: esta célula leva ~30-45 min na primeira execução (download de 141M footprints via DuckDB httpfs)
# Em execuções seguintes com banco persistente, DuckDB usa cache local e é instantâneo

n = ddb.carregar_pais(con, country_iso='BRA', tabela='footprints_bra')
print(f'Carregados {n:,} footprints do Brasil')

In [ ]:
# Carrega AOI como tabela DuckDB (ST_Read do GeoJSON local)
ddb.carregar_aoi_geojson(con, cambuquira.path, 'aoi_cambuquira')

# Recorte espacial via ST_Intersection — a operação central
n_recorte = ddb.recortar_por_aoi(
    con,
    tabela_footprints='footprints_bra',
    tabela_aoi='aoi_cambuquira',
    tabela_recorte='recorte_cambuquira',
)
print(f'{n_recorte:,} edificações dentro de Cambuquira/MG')

In [ ]:
# Análise agregada: contagem por fonte, área total, área média
_, miny, _, maxy = cambuquira.bounds
resumo = analysis.resumo_aoi(
    con, 'recorte_cambuquira',
    aoi_nome=cambuquira.nome_exibicao,
    lat_media=(miny + maxy) / 2,  # ajuste de precisão para cálculo de área
)

# Imprime em formato legível
print(analysis.formatar_resumo_texto(resumo))

In [ ]:
# Resumo também é serializável em JSON para integração com outras ferramentas
import json
print(json.dumps(resumo.to_dict(), indent=2, ensure_ascii=False))

## 5. Comparação entre AOIs

Mesmo pipeline aplicado a outra AOI permite comparações empíricas.

In [ ]:
# Processa Uberlândia (reusa footprints_bra já carregado)
uberlandia = aoi_mod.carregar_aoi('uberlandia_mg')
ddb.carregar_aoi_geojson(con, uberlandia.path, 'aoi_uberlandia')
ddb.recortar_por_aoi(
    con,
    tabela_footprints='footprints_bra',
    tabela_aoi='aoi_uberlandia',
    tabela_recorte='recorte_uberlandia',
)
print('Uberlândia recortada')

In [ ]:
# Comparar métricas lado a lado
cmp = analysis.comparar_aois(
    con,
    tabelas_por_aoi={
        'Cambuquira/MG': 'recorte_cambuquira',
        'Uberlândia/MG': 'recorte_uberlandia',
    },
    bounds_por_aoi={
        'Cambuquira/MG': cambuquira.bounds,
        'Uberlândia/MG': uberlandia.bounds,
    },
)
print(analysis.formatar_comparacao_texto(cmp))

## 6. Exportação e visualização

Export para GeoJSON ou FlatGeobuf, ou geração do webmap HTML autocontido.

In [ ]:
# Export GeoJSON do recorte (para QGIS, Leaflet, etc.)
config.ensure_data_dirs()
ddb.exportar_geojson(
    con,
    tabela='recorte_cambuquira',
    output_path=config.PROCESSED_DATA_DIR / 'cambuquira_mg.geojson',
)

In [ ]:
# Gerar webmap multi-AOI com dashboard analítico
webmap.gerar_webmap_multi(
    con,
    aois_e_tabelas=[
        (cambuquira, 'recorte_cambuquira'),
        (uberlandia, 'recorte_uberlandia'),
    ],
    output_dir=config.WEBMAP_DIR,
    min_area_m2=30,       # descarta footprints < 30 m² (para Uberlândia com 538k polígonos)
    simplify_tolerance=1e-5,  # simplifica geometrias (~1m em WGS84)
)
print(f'Webmap gerado em {config.WEBMAP_DIR / "index.html"}')
print('Para visualizar: python -m http.server 8000 --directory webmap')

In [ ]:
# Fechar conexão (importante se usando banco persistente)
con.close()
print('[OK] Pipeline completo. Confira webmap/index.html')

## Próximos passos

- Experimentar com outra AOI: use `obr-explorer aoi fetch --codigo <IBGE>` para baixar
  o polígono oficial de qualquer município brasileiro via API IBGE
- Rodar queries SQL customizadas direto na conexão DuckDB — ela tem toda a flexibilidade
  de SQL + funções espaciais `ST_*`
- Integrar o webmap gerado via GitHub Pages (ver seção no `README.md`)
- Ler os ADRs em `docs/DECISIONS.md` para entender por que cada peça foi feita assim

---

Projeto: [github.com/PedroLuizskt/open-buildings-explorer-br](https://github.com/PedroLuizskt/open-buildings-explorer-br)